# ViT-Large + DINOv3 - amunisi submission #3
Butuh `pipeline.py` **v20** (fallback SSL utk DINOv3).

**Dua opsi setup - jalankan SALAH SATU:**
- **Opsi A (akun sendiri):** sel `A1` (mount+extract) + `A2` (pipeline dari Drive).
- **Opsi B (akun pinjaman, tanpa mount):** sel `B1` (gdown) saja. Wajib lanjut sel `B2` (rclone) utk persistensi!

| Run | GPU | Estimasi |
|---|---|---|
| Smoke DINOv3 | apa saja | ~5 mnt |
| DINOv3-B @224, 5-fold | L4 | ~2-3 jam |
| ViT-L @384, 5-fold | L4 bs8 / A100 bs16 | ~8-10 jam (L4) / ~4 jam (A100) |

Resume-aware: quota putus -> Run ulang, fold selesai auto-skip (Opsi B: tarik dulu via rclone).

In [ ]:
# Extract dataset dari Drive ke disk lokal Colab
import os, glob
if os.path.exists('/content'):
    ARCHIVE = '/content/drive/MyDrive/SATDAT.rar'
    from google.colab import drive; drive.mount('/content/drive')
    os.makedirs('/content/data', exist_ok=True)
    if not glob.glob('/content/data/**/train', recursive=True):
        if ARCHIVE.endswith('.rar'):
            !apt-get -qq install -y unrar >/dev/null
            !unrar x -o+ "$ARCHIVE" /content/data/
        else:
            !unzip -q "$ARCHIVE" -d /content/data
    train_dir = glob.glob('/content/data/**/train', recursive=True)[0]
    os.chdir(os.path.dirname(train_dir))
    print('cwd:', os.getcwd(), '| train:', os.path.exists('train'), '| test:', os.path.exists('test'))

In [ ]:
# Install keras-hub + pipeline terbaru
!pip install -q keras-hub
import importlib
if os.path.exists('/content'):
    !cp /content/drive/MyDrive/pipeline.py .
import pipeline
importlib.reload(pipeline)
print('pipeline version:', pipeline.__version__)
assert pipeline.__version__ >= '20', 'butuh pipeline v20 (fallback SSL DINOv3)'

In [ ]:
# ==== B1. SETUP TANPA MOUNT (akun pinjaman): gdown data + pipeline ====
# Prasyarat: SATDAT.rar & pipeline.py di Drive-mu di-set "Anyone with the link".
# ID = bagian antara /d/ dan /view di link share.
!pip install -q gdown keras-hub

ID_RAR  = 'PASTE_ID_SATDAT_RAR'    # <- ganti
ID_PIPE = 'PASTE_ID_PIPELINE_PY'   # <- ganti

import os, glob

if not os.path.exists('/content/SATDAT.rar'):
    !gdown $ID_RAR -O /content/SATDAT.rar

!apt-get -qq install -y unrar > /dev/null
os.makedirs('/content/data', exist_ok=True)
if not glob.glob('/content/data/**/train', recursive=True):
    !unrar x -o+ /content/SATDAT.rar /content/data/ > /dev/null

train_dir = glob.glob('/content/data/**/train', recursive=True)[0]
os.chdir(os.path.dirname(train_dir))

!gdown $ID_PIPE -O pipeline.py
import importlib, pipeline
importlib.reload(pipeline)

print('cwd:', os.getcwd())
print('train:', os.path.exists('train'), '| test:', os.path.exists('test'))
print('pipeline version:', pipeline.__version__)
assert pipeline.__version__ >= '20', 'pipeline.py di Drive belum v20 - timpa (Manage versions)'

In [ ]:
# ==== B2. RCLONE: persistensi hasil ke Drive-mu (HANYA utk sesi TANPA mount) ====
import os, subprocess

if os.path.exists('/content/drive/MyDrive'):
    raise SystemExit('Drive TER-MOUNT di sesi ini -> B2 TIDAK PERLU. Lewati sel ini; '
                     'training menulis langsung ke Drive.')

assert os.path.exists('/content/rclone.conf'), (
    'rclone.conf BELUM diupload! Panel Files -> upload dari laptop: '
    'C:/Users/LEGION/Downloads/rclone.conf -> ke /content. Baru run ulang sel ini.')

!apt-get -qq install -y rclone > /dev/null
os.makedirs(os.path.expanduser('~/.config/rclone'), exist_ok=True)
!cp /content/rclone.conf ~/.config/rclone/rclone.conf

# uji koneksi - gagal = berhenti di sini, bukan diam-diam
rc = os.system('rclone lsd gdrive: > /dev/null 2>&1')
assert rc == 0, 'rclone tidak bisa akses gdrive: - conf salah/expired'
print('koneksi gdrive OK')

# tarik hasil sesi sebelumnya (resume)
!rclone copy gdrive:satria/experiments_dinov3 /content/exp_dinov3 -q
!rclone copy gdrive:satria/experiments_vitl  /content/exp_vitl  -q

# sync latar belakang tiap 5 menit
subprocess.Popen(
    'while true; do '
    'rclone copy /content/exp_dinov3 gdrive:satria/experiments_dinov3 -q; '
    'rclone copy /content/exp_vitl  gdrive:satria/experiments_vitl  -q; '
    'sleep 300; done', shell=True)
print('rclone siap + sync latar belakang jalan (tiap 5 mnt)')
# Akhir sesi (akun pinjaman): !rm /content/rclone.conf ~/.config/rclone/rclone.conf

In [ ]:
# ==== 0. DIAGNOSTIK DINOv3 (kenapa smoke = acak 0.1667?) - 2 mnt, tanpa training ====
!pip install -q keras-hub
import tensorflow as tf, numpy as np, keras_hub
tf.keras.mixed_precision.set_global_policy('float32')   # fp16 OFF dulu (uji tersangka NaN)

PRESET = 'dinov3_vit_base_lvd1689m'   # ganti ke _large kalau mau uji ViT-L DINO

# 1. jalur mana? ImageClassifier ada atau fallback?
try:
    m = keras_hub.models.ImageClassifier.from_preset(PRESET, num_classes=3)
    print('>>> ImageClassifier BERHASIL - fallback TIDAK dipakai. type:', type(m).__name__)
except Exception as e:
    print('>>> ImageClassifier GAGAL -> fallback dipakai:', str(e)[:160])

# 2. output backbone: struktur + apakah BERUBAH antar gambar beda (tes kunci)
bb = keras_hub.models.Backbone.from_preset(PRESET)
a = tf.random.uniform([1, 224, 224, 3])
b = tf.random.uniform([1, 224, 224, 3])
oa, ob = bb(a), bb(b)

def info(o):
    if isinstance(o, dict):
        return {k: tuple(v.shape) for k, v in o.items()}
    return tuple(o.shape)
print('struktur output backbone:', info(oa))

ta = list(oa.values())[-1] if isinstance(oa, dict) else oa
tb = list(ob.values())[-1] if isinstance(ob, dict) else ob
ta, tb = tf.cast(ta, tf.float32), tf.cast(tb, tf.float32)
print('std fitur (harus > 0.1):        ', float(tf.math.reduce_std(ta)))
print('beda antar 2 gambar (harus >0.3):', float(tf.reduce_mean(tf.abs(ta - tb))))
print('ada NaN?                        ', bool(tf.reduce_any(tf.math.is_nan(ta))))

# kembalikan policy semula
tf.keras.mixed_precision.set_global_policy('float32')

In [ ]:
# ==== 1. SMOKE TEST DINOv3 (WAJIB sebelum train penuh - jalur baru, ~5 mnt) ====
cfg_smoke = {
    'backbone': 'dinov3_vit_base_lvd1689m',
    'img_size': 224,
    'batch_size': 16,
    'warmup_epochs': 0,
    'finetune': 'full',
    'mixed_precision': True,
    'smoke': True,                  # 300 gambar, 1 epoch: uji end-to-end build->train->save
    'out_root': '/content/smoke',   # sampah, dibuang
}
pipeline.run(cfg_smoke)
print('SMOKE DINOv3 LOLOS - lanjut train penuh')

In [ ]:
# ==== 2. TRAIN DINOv3-B @224, 5-fold ====
MOUNTED = os.path.exists('/content/drive/MyDrive')
cfg_dino = {
    'backbone': 'dinov3_vit_base_lvd1689m',
    'img_size': 224,
    'batch_size': 32,               # OOM? turunkan 16
    'epochs': 14,
    'lr': 1e-5,
    'warmup_epochs': 3,             # PENTING: head dari nol (SSL tak punya head)
    'finetune': 'full',
    'label_smoothing': 0.1,
    'use_class_weight': True,
    'augment': True,
    'random_erase': 0.25,
    'mixed_precision': True,
    'cv_folds': 5,
    # Opsi A (mount): langsung ke Drive. Opsi B: lokal + rclone sync (sel B2!)
    'out_root': ('/content/drive/MyDrive/satria/experiments_dinov3' if MOUNTED
                 else '/content/exp_dinov3'),
}
print('out_root:', cfg_dino['out_root'],
      '' if MOUNTED else '(LOKAL - pastikan sel B2 rclone sudah jalan!)')
pipeline.run_cv(cfg_dino)

In [ ]:
# ==== 3. TRAIN ViT-L @384, 5-fold (berat! idealnya A100) ====
MOUNTED = os.path.exists('/content/drive/MyDrive')
cfg_vitl = {
    'backbone': 'vit_large_patch16_384_imagenet',
    'img_size': 384,
    'batch_size': 8,                # A100: 16
    'epochs': 10,                   # model besar konvergen cepat; early-stop yang motong
    'lr': 1e-5,                     # kalau loss warmup->utama tak turun mulus: coba 5e-6
    'warmup_epochs': 2,
    'finetune': 'full',
    'label_smoothing': 0.1,
    'use_class_weight': True,
    'augment': True,
    'random_erase': 0.25,
    'mixed_precision': True,
    'cv_folds': 5,
    'out_root': ('/content/drive/MyDrive/satria/experiments_vitl' if MOUNTED
                 else '/content/exp_vitl'),
}
print('out_root:', cfg_vitl['out_root'],
      '' if MOUNTED else '(LOKAL - pastikan sel B2 rclone sudah jalan!)')
pipeline.run_cv(cfg_vitl)

In [ ]:
# ==== 4. OOF model baru (jalankan setelah masing2 train selesai) ====
import numpy as np
MOUNTED = os.path.exists('/content/drive/MyDrive')
# Opsi A: simpan ke Drive. Opsi B: simpan ke folder exp lokal (ikut tersync rclone).
DRV = '/content/drive/MyDrive/satria' if MOUNTED else None

p_dino, y = pipeline.oof_predictions(cfg_dino)
np.save((DRV or cfg_dino['out_root']) + '/oof_probs_dino.npy', p_dino)
pipeline.calibration_report(p_dino, y)

p_vitl, y = pipeline.oof_predictions(cfg_vitl)
np.save((DRV or cfg_vitl['out_root']) + '/oof_probs_vitl.npy', p_vitl)
pipeline.calibration_report(p_vitl, y)

In [ ]:
# ==== 5. SIMULASI KOMPOSISI FINAL (semua kandidat, di OOF - penentu submission #3) ====
import numpy as np
from sklearn.metrics import f1_score
DRV = '/content/drive/MyDrive/satria'

def L(nama):
    p = np.load(DRV + '/' + nama)
    return p / p.sum(1, keepdims=True)

pv, pt = L('oof_probs_s123.npy'), L('oof_probs_vit.npy')
pl, pd_ = L('oof_probs_vitl.npy'), L('oof_probs_dino.npy')
y = np.load(DRV + '/oof_labels_s123.npy')

combos = [
    ('ViT-L solo',            pl),
    ('DINOv3 solo',           pd_),
    ('baseline #1: V2S+ViT',  pv + pt),
    ('V2S+ViT+ViTL',          pv + pt + pl),
    ('V2S+ViT+DINO',          pv + pt + pd_),
    ('V2S+ViTL',              pv + pl),
    ('ViT+ViTL+DINO',         pt + pl + pd_),
    ('SEMUA (4)',             pv + pt + pl + pd_),
]
for nama, p in combos:
    f1 = f1_score(y, p.argmax(1), average='macro')
    print(f"{nama:24s} macro F1 = {f1:.4f}")